# 基于HCCL的稀疏图分片与全局度数统计

图数据通常具有顶点邻接长度不规则、热点顶点连接数高等特点。CSR压缩邻接表使用`row_ptr`和`col_indices`保存图的邻接关系，能够减少存储开销并支持按顶点扫描；分布式执行还需要把顶点或边分配给不同Rank，并将各Rank产生的局部度数贡献汇总为全局结果。

本实验基于C++17、CMake和Ascend HCCL实现CSR图分片与全局度数统计。工程提供`simulate`和`hccl`两种后端：前者在单进程中模拟多个Rank，用于检查图生成、分片、局部统计和集合通信语义；后者在双卡或多卡服务器上启动多个进程，使用真实HCCL执行ReduceScatter和AllGather。实验比较`vertex_range`、`edge_hash`和`degree_balance`三种分片策略，并从邻接项负载、全局Top-K、通信缓冲区和通信耗时等方面进行分析。

本节学习大纲如下：

1. 实验概述：介绍实验目标、前置知识和实验要点；
2. 环境准备：创建工程目录并检查C++17/CMake工具链与CANN条件；
3. 问题分析：分析CSR存储、图分片、本地度数统计和HCCL集合通信流程；
4. 核心程序开发：实现配置、图生成、分片、度数统计、模拟通信、HCCL后端和结果报告；
5. 结果验证与性能分析：运行模拟和双Rank HCCL实验，比较三种策略并解释通信时间波动；
6. 实验总结：归纳CSR图分片和全局度数统计的设计要点。


---
## 1. 实验概述

本实验以服务调用依赖图为应用背景，将服务实例或服务模块抽象为顶点，将调用、依赖或通信关系抽象为无向边。图首先转换为CSR邻接表，再按照不同策略分配给多个逻辑Rank。每个Rank只根据自己获得的顶点或边计算局部度数贡献向量，随后使用ReduceScatter完成全局求和与分片，使用AllGather恢复完整全局度数向量。

实验默认使用1024个顶点和4000条无向边，因而CSR邻接项数为8000。`uniform`图用于观察连接较均匀时的分片表现，`skewed`图将较多边连接到低编号热点顶点，用于观察顶点范围分片的负载倾斜以及度数均衡策略的改善。


### 1.1 实验目标

完成本实验后应达到以下目标：

1. 理解图、顶点度数和CSR压缩邻接表之间的关系，能够根据`row_ptr`和`col_indices`解释一条邻接表的存储位置；
2. 掌握`vertex_range`、`edge_hash`和`degree_balance`三种分片策略，理解顶点数量均衡与邻接项处理量均衡之间的区别；
3. 掌握双Rank HCCL流程，包括本地度数贡献生成、ReduceScatter分片归约、AllGather全局恢复和串行参考校验；
4. 具备根据分片负载、热点顶点、缓冲区规模、通信耗时和正确性结果分析实验现象的能力。


### 1.2 前置知识

本实验要求提前具备以下基础：

1. 图数据结构：理解顶点、边、度数以及无向图中一条边对两个端点各贡献一次；
2. CSR存储：理解`row_ptr[v]`到`row_ptr[v+1]`对应顶点`v`的连续邻接区间；
3. 分布式分片：理解Rank、world size、本地数据和全局结果的关系；
4. 集合通信：理解ReduceScatter的“求和后等长分片”和AllGather的“收集所有分片”；
5. C++与CMake基础：能够阅读C++17头文件、源文件、CMake配置和Shell脚本；
6. Ascend环境基础：了解CANN环境变量、NPU设备编号和多进程HCCL启动方式。


### 1.3 实验要点

实验中应重点关注以下内容：

1. CSR结构：一条无向边在CSR中展开为两个方向，因此`col_indices`长度为边数的两倍；
2. 分片单位：`vertex_range`和`degree_balance`分配完整顶点邻接行，`edge_hash`分配原始边；
3. 负载指标：顶点数或边数是分片单元数量，邻接项数才更接近实际CSR扫描工作量；
4. 通信布局：HCCL需要各Rank输入等长向量，因此顶点数不足world size倍数时要补零；
5. 正确性优先：`result=PASS`且`mismatch_count=0`是全局度数统计正确的必要检查；
6. 性能解释：小规模通信耗时容易受HCCL初始化、进程同步和设备同步影响，不能仅凭单次时间判断策略优劣。


---
## 2. 环境准备

### 2.1 创建实验目录并检查编译环境

Notebook会在`src/03.04_extra_hccl_csr_graph_partition`目录下生成一份可独立编译运行的工程。工程包含CPU模拟后端和真实HCCL后端；普通Notebook环境至少可以构建并运行`simulate`，双Rank HCCL运行需要已安装CANN且具有两张可见NPU。

目录划分如下：

- `include/`：保存实验配置、CSR图、分片、度数、集合通信、HCCL后端、指标和结果接口；
- `src/`：保存图生成、分片、度数统计、通信后端、指标、结果报告和主程序；
- `scripts/`：保存模拟后端与HCCL后端的构建运行脚本；
- `results/`：保存模拟输出、HCCL Rank日志和精简结果文件。


In [ ]:
from pathlib import Path
import os
import subprocess

WORK_DIR = Path("src/03.04_extra_hccl_csr_graph_partition").resolve()
for directory in [WORK_DIR / "include", WORK_DIR / "src", WORK_DIR / "scripts", WORK_DIR / "results"]:
    directory.mkdir(parents=True, exist_ok=True)

print("Experiment directory:", WORK_DIR)
print("Current platform:", os.name)
subprocess.run(["c++", "--version"], check=False)
subprocess.run(["cmake", "--version"], check=False)
print("CANN/HCCL is required only for the real hccl backend.")


### 2.2 写入工程公共接口

本节写入实验配置、CSR图、分片结果、度数向量、通信结果、HCCL运行参数、负载指标和结果报告接口。实现文件将在第4节写入，CMake构建文件和两个运行脚本将在第5节写入。


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/include/experiment_config.hpp
#pragma once

#include <cstdint>
#include <string>

namespace hccl_csr_graph {

enum class Backend {
    // simulate 在单进程中模拟多 Rank，hccl 使用真实 Ascend HCCL 多进程通信。
    Simulate,
    Hccl,
};

struct ExperimentConfig {
    Backend backend = Backend::Simulate;
    // world_size 是通信域总 Rank 数，rank 是当前进程编号，local_device 是本机 NPU 编号。
    int world_size = 4;
    int rank = 0;
    int local_device = 0;
    int vertex_count = 1024;
    int edge_count = 4000;
    // graph_mode 控制均匀、热点倾斜或簇状图结构。
    std::string graph_mode = "skewed";
    // vertex_range、edge_hash 或 degree_balance。
    std::string partition_strategy = "degree_balance";
    std::uint32_t seed = 2026;
    int top_k = 10;
    // rank0 写入 HCCL RootInfo，其他 Rank 读取后加入同一个通信域。
    std::string root_info_file = "/tmp/hccl_csr_graph_partition_root.info";
};

Backend parse_backend(const std::string& value);
std::string backend_name(Backend backend);
void validate_config(const ExperimentConfig& config);

}  // namespace hccl_csr_graph


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/include/csr_graph.hpp
#pragma once

#include <vector>

#include "experiment_config.hpp"

namespace hccl_csr_graph {

// 无向边只保存一次；构造 CSR 时会分别写入 src 和 dst 的邻接表。
struct Edge {
    int src = 0;
    int dst = 0;
};

// CSR 图：row_ptr 给出每个顶点邻接区间，col_indices 连续保存邻居编号。
struct CsrGraph {
    int vertex_count = 0;
    // 原始无向边列表，用于 edge_hash 分片和内存估算。
    std::vector<Edge> edges;
    std::vector<int> row_ptr;
    std::vector<int> col_indices;

    int edge_count() const;
    int adjacency_count() const;
    int degree(int vertex) const;
};

// 按 ExperimentConfig 生成图并转换为 CSR 存储。
CsrGraph generate_graph(const ExperimentConfig& config);

}  // namespace hccl_csr_graph


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/include/partition.hpp
#pragma once

#include <string>
#include <vector>

#include "csr_graph.hpp"

namespace hccl_csr_graph {

// 一次图分片结果，同时保存分片内容和每个 Rank 的工作量统计。
struct PartitionResult {
    std::string strategy;
    // edge_based=true 表示分配边，否则分配顶点及其整行 CSR 邻接表。
    bool edge_based = false;
    std::vector<std::vector<int>> vertex_ids_by_rank;
    std::vector<std::vector<int>> edge_ids_by_rank;
    // item_counts 统计顶点数或边数；adjacency_counts 统计实际邻接项处理量。
    std::vector<int> item_counts;
    std::vector<int> adjacency_counts;
};

// 根据指定策略把图划分到 world_size 个逻辑 Rank。
PartitionResult partition_graph(const CsrGraph& graph, int world_size, const std::string& strategy);

}  // namespace hccl_csr_graph


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/include/degree.hpp
#pragma once

#include <cstddef>
#include <cstdint>
#include <utility>
#include <vector>

#include "csr_graph.hpp"
#include "partition.hpp"

namespace hccl_csr_graph {

using Count = std::int64_t;
using CountVector = std::vector<Count>;

// 集合通信要求每个 Rank 接收等长分片，因此将顶点数补齐到 world_size 的倍数。
struct PaddingInfo {
    std::size_t padded_size = 0;
    std::size_t shard_size = 0;
};

// 应用层通信向量和图存储结构的字节数估算。
struct BufferEstimate {
    std::size_t input_degree_vector_bytes_per_rank = 0;
    std::size_t reduce_scatter_shard_bytes_per_rank = 0;
    std::size_t all_gather_result_bytes_per_rank = 0;
    std::size_t all_ranks_input_bytes_total = 0;
    std::size_t csr_row_ptr_bytes = 0;
    std::size_t csr_col_indices_bytes = 0;
    std::size_t edge_list_bytes = 0;
};

PaddingInfo compute_padding(std::size_t vector_size, int world_size);
// 只根据当前 Rank 获得的顶点或边分片生成局部度数贡献向量。
CountVector compute_local_degree(const CsrGraph& graph, const PartitionResult& partition, int rank);
// 直接扫描完整 CSR 图得到串行标准答案。
CountVector serial_reference_degree(const CsrGraph& graph);
CountVector pad_vector(const CountVector& values, std::size_t padded_size);
CountVector unpad_vector(const CountVector& values, std::size_t original_size);
std::pair<bool, std::vector<std::size_t>> verify_result(
    const CountVector& global_degree,
    const CountVector& reference_degree);
// 估算通信缓冲区、CSR 数组和原始边表占用。
BufferEstimate estimate_buffers(
    std::size_t padded_vertex_count,
    std::size_t shard_size,
    int world_size,
    const CsrGraph& graph);

}  // namespace hccl_csr_graph


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/include/collectives.hpp
#pragma once

#include <cstddef>
#include <vector>

#include "degree.hpp"

namespace hccl_csr_graph {

// ReduceScatter 和 AllGather 的统一输出，供模拟后端和 HCCL 后端共用。
struct CommunicationResult {
    std::size_t padded_vertex_count = 0;
    std::size_t shard_size = 0;
    // 对全局度数求和后，每个 Rank 获得的连续顶点分片。
    std::vector<CountVector> reduced_shards;
    // AllGather 后每个 Rank 恢复出的完整全局度数向量。
    CountVector gathered_degree;
    double reduce_scatter_ms = 0.0;
    double all_gather_ms = 0.0;
};

// CPU 模拟版集合通信：逐位置求和、等长切片，再拼接完整向量。
CommunicationResult run_simulated_collectives(
    const std::vector<CountVector>& local_degrees,
    int world_size);

}  // namespace hccl_csr_graph


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/include/hccl_backend.hpp
#pragma once

#include <string>

#include "collectives.hpp"

namespace hccl_csr_graph {

// 当前 HCCL 进程的全局 Rank、通信规模、本地 NPU 和 RootInfo 文件。
struct HcclRuntimeOptions {
    int rank = 0;
    int world_size = 1;
    int local_device = 0;
    std::string root_info_file;
};

bool is_hccl_compiled();
// 在设备内存中执行真实 HCCL ReduceScatter 和 AllGather。
CommunicationResult run_hccl_collectives(
    const CountVector& local_degree,
    std::size_t vertex_count,
    const HcclRuntimeOptions& options);

}  // namespace hccl_csr_graph


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/include/metrics.hpp
#pragma once

#include <utility>
#include <vector>

#include "degree.hpp"
#include "partition.hpp"

namespace hccl_csr_graph {

// 同时描述“分片单元数量”和“邻接项计算量”的负载均衡情况。
struct PartitionMetrics {
    int min_items = 0;
    int max_items = 0;
    double avg_items = 0.0;
    double item_imbalance = 0.0;
    int min_adjacency = 0;
    int max_adjacency = 0;
    double avg_adjacency = 0.0;
    double adjacency_imbalance = 0.0;
};

// 不均衡度使用最大值 / 平均值，越接近 1 越均衡。
PartitionMetrics compute_partition_metrics(const PartitionResult& partition);
// 按度数降序返回全局或局部 Top-K 顶点。
std::vector<std::pair<int, Count>> topk_vertices(const CountVector& degree, int top_k);

}  // namespace hccl_csr_graph


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/include/result_report.hpp
#pragma once

#include <string>
#include <vector>

#include "collectives.hpp"
#include "csr_graph.hpp"
#include "degree.hpp"
#include "experiment_config.hpp"
#include "metrics.hpp"
#include "partition.hpp"

namespace hccl_csr_graph {

// 汇总实验从图生成、分片、局部计算、集合通信到正确性验证的全部结果。
struct ExperimentResult {
    ExperimentConfig config;
    CsrGraph graph;
    PartitionResult partition;
    std::vector<PartitionResult> strategy_results;
    // 每个 Rank 对全局顶点空间产生的局部度数贡献向量。
    std::vector<CountVector> local_degrees;
    CommunicationResult communication;
    CountVector global_degree;
    CountVector reference_degree;
    bool correct = false;
    std::vector<std::size_t> mismatches;
    BufferEstimate buffers;
};

// 打印配置、CSR、分片、通信、Top-K、缓冲区和正确性表格。
void print_results(const ExperimentResult& result);
// 写入便于 Notebook 和实验报告读取的精简结果文件。
void write_result_summary(const std::string& output_path, const ExperimentResult& result);

}  // namespace hccl_csr_graph


### 2.3 工程公共接口检查

公共头文件写入完成后，检查关键文件是否已经生成。检查结果均为`OK`时，说明图数据、分片、通信和结果报告接口已经就绪。


In [ ]:
required_headers = [
    "include/experiment_config.hpp",
    "include/csr_graph.hpp",
    "include/partition.hpp",
    "include/degree.hpp",
    "include/collectives.hpp",
    "include/hccl_backend.hpp",
    "include/metrics.hpp",
    "include/result_report.hpp",
]

for relative_path in required_headers:
    path = WORK_DIR / relative_path
    print(f"{relative_path}: {'OK' if path.exists() else 'MISSING'}")


---
## 3. 问题分析

本节分析CSR压缩存储、图生成模式、三种分片策略、局部度数向量、补齐规则、ReduceScatter与AllGather以及真实HCCL多进程流程。后续C++实现将围绕这些数据结构和通信语义展开。


### 3.1 图数据与CSR压缩存储

设图有$V$个顶点和$E$条无向边。CSR使用长度为$V+1$的`row_ptr`和长度为$2E$的`col_indices`保存无向图：边$(u,v)$会在顶点$u$的邻接行中写入$v$，并在顶点$v$的邻接行中写入$u$。顶点$v$的邻接区间为：

$$
neighbors(v)=colIndices[rowPtr[v]:rowPtr[v+1])
$$

因此：

$$
degree(v)=rowPtr[v+1]-rowPtr[v]
$$

CSR不需要为每个顶点单独分配可变长度数组，邻接项连续存储，适合按顶点范围扫描和统计。


### 3.2 图生成模式与热点顶点

工程使用固定随机种子生成无自环、无重复边的简单无向图：

- `uniform`：两个端点在全体顶点中均匀采样，连接较分散；
- `skewed`：端点以较高概率从低编号热点顶点中采样，形成高度数顶点；
- `clustered`：端点优先从同一簇中采样，模拟模块化依赖关系。

实验中`vertex_count=1024`、`edge_count=4000`时，平均度数为：

$$
averageDegree=\frac{2E}{V}=\frac{8000}{1024}\approx7.812
$$

倾斜图的最大度数远高于平均度数，适合观察连续顶点范围分片的热点集中现象。


### 3.3 三种图分片策略

1. `vertex_range`：按顶点编号连续切分，每个Rank获得一段完整CSR邻接行。顶点数容易均衡，但热点集中时邻接项负载可能不均衡；
2. `edge_hash`：对原始边编号或边内容做哈希，把边分散到不同Rank。边数和邻接项通常均衡，但同一顶点的度数贡献可能分散，依赖全局规约合并；
3. `degree_balance`：按顶点度数从高到低，将当前顶点分配给累计邻接项最少的Rank。它不保证顶点数量完全相等，但目标是让CSR扫描量尽可能均衡。

用最大负载与平均负载定义不均衡率：

$$
imbalance=\frac{maxLoad}{avgLoad}
$$

不均衡率越接近1，表示对应统计口径下越均衡。分析时应分别查看`item_imbalance`和`adjacency_imbalance`。


### 3.4 局部度数贡献向量

每个Rank都构造长度为$V$的`int64`局部向量$localDegree_r$。如果当前Rank负责顶点，则直接扫描对应CSR行并增加该顶点度数；如果当前Rank负责边，则对边的两个端点各增加一次：

$$
localDegree_r[v]=\sum_{e\in partition_r} contribution(e,v)
$$

所有Rank局部向量按位置相加后得到全局顶点度数向量。`edge_hash`下同一顶点可能在多个Rank产生贡献，正是ReduceScatter需要执行求和的原因。


### 3.5 向量补齐与通信分片

ReduceScatter要求通信向量能被Rank数等分。设原始顶点数为$V$，world size为$P$，补齐后的长度为：

$$
V'=\left\lceil\frac{V}{P}\right\rceil P
$$

每个Rank的分片长度为：

$$
shardSize=\frac{V'}{P}
$$

补齐位置使用0，不影响原始顶点的度数统计。AllGather恢复完整长度$V'$的向量后，再去掉补齐尾部，得到长度为$V$的全局结果。


### 3.6 ReduceScatter与AllGather流程

设Rank数为2，补齐后的度数向量长度为1024。ReduceScatter先对所有Rank的局部向量逐位置求和，再把结果切成两个长度为512的连续分片：

$$
shard_r=\operatorname{sum}_{q=0}^{P-1}(localDegree_q)[r\cdot shardSize:(r+1)\cdot shardSize]
$$

AllGather再把各Rank持有的分片收集到每个Rank，恢复完整全局度数向量。结果应满足各分片度数和之和等于CSR邻接项总数$2E$，并与串行CSR扫描结果逐元素一致。


### 3.7 模拟后端与真实HCCL多进程

`simulate`后端在一个进程中生成全部Rank的局部向量，逐位置执行求和和拼接，便于在没有NPU的环境中检查应用层逻辑。`hccl`后端则由脚本启动多个进程：每个进程读取`RANK`、`WORLD_SIZE`和`LOCAL_RANK`，独立生成相同图和分片，只把当前Rank的局部向量送入真实HCCL通信域。

Rank 0写入HCCL RootInfo文件，其他Rank读取该文件并加入同一通信域。所有进程必须调用相同顺序的集合通信，只有Rank 0负责打印汇总结果和写精简结果文件。


### 3.8 缓冲区与通信耗时指标

默认双Rank配置下，输入向量为1024个`int64`，每个Rank输入缓冲区大小为：

$$
1024\times8=8192\ \text{bytes}
$$

ReduceScatter后每个Rank保留512个`int64`，即4096 bytes；AllGather后恢复8192 bytes。当前消息规模较小，单次通信时间更容易受到HCCL运行时、首次调用、进程同步和设备同步影响，因此应把耗时作为观测值，并通过重复实验统计均值和波动范围。


### 3.9 实验参数设置

默认实验使用`vertex_count=1024`、`edge_count=4000`、`world_size=2`、`seed=2026`和`top_k=10`。最终双Rank对照实验固定图规模和随机种子，只改变图模式与当前选用的分片策略，保证不同组之间可以直接比较。


In [ ]:
VERTEX_COUNT = 1024
EDGE_COUNT = 4000
WORLD_SIZE = 2
SEED = 2026
TOP_K = 10
STRATEGIES = ["vertex_range", "edge_hash", "degree_balance"]

print("vertexCount:", VERTEX_COUNT)
print("edgeCount:", EDGE_COUNT)
print("worldSize:", WORLD_SIZE)
print("adjacencyCount:", 2 * EDGE_COUNT)
print("paddedVertexCount:", ((VERTEX_COUNT + WORLD_SIZE - 1) // WORLD_SIZE) * WORLD_SIZE)
print("shardSize:", VERTEX_COUNT // WORLD_SIZE)
print("averageDegree:", 2 * EDGE_COUNT / VERTEX_COUNT)
print("strategies:", STRATEGIES)


需要注意，`item_imbalance`和`adjacency_imbalance`回答的是不同问题：前者衡量分片顶点或边数量是否均衡，后者衡量实际CSR邻接项工作量是否均衡。在热点图上，`vertex_range`可能达到`adjacency_imbalance=1.417`，而`degree_balance`和`edge_hash`可以达到1.000；这不代表三种策略的语义完全相同。


---
## 4. 核心程序开发

本节依次实现配置解析、图生成与CSR构建、分片、局部度数、模拟集合通信、真实HCCL后端、指标统计、结果报告和主程序。所有实现均与交付版C++工程保持一致。


### 4.1 实验配置与CSR图生成

`ExperimentConfig`集中保存后端、Rank规模、设备编号、图规模、图模式、分片策略和RootInfo路径。`generate_graph`生成确定性简单无向图并构造CSR；`degree(v)`直接由`row_ptr[v+1]-row_ptr[v]`得到。


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/src/experiment_config.cpp
#include "experiment_config.hpp"

#include <stdexcept>
#include <string>

namespace hccl_csr_graph {

Backend parse_backend(const std::string& value) {
    if (value == "simulate") {
        return Backend::Simulate;
    }
    if (value == "hccl") {
        return Backend::Hccl;
    }
    throw std::invalid_argument("unsupported backend: " + value);
}

std::string backend_name(Backend backend) {
    switch (backend) {
        case Backend::Simulate:
            return "simulate";
        case Backend::Hccl:
            return "hccl";
    }
    return "unknown";
}

void validate_config(const ExperimentConfig& config) {
    // 在图生成和通信前统一检查规模、Rank 范围、图模式及分片策略。
    if (config.world_size <= 0) {
        throw std::invalid_argument("world_size must be positive");
    }
    if (config.rank < 0 || config.rank >= config.world_size) {
        throw std::invalid_argument("rank must be in [0, world_size)");
    }
    if (config.local_device < 0) {
        throw std::invalid_argument("local_device must be non-negative");
    }
    if (config.vertex_count <= 1) {
        throw std::invalid_argument("vertex_count must be greater than 1");
    }
    if (config.edge_count < 0) {
        throw std::invalid_argument("edge_count must be non-negative");
    }
    // 简单无向图最多包含 V * (V - 1) / 2 条边。
    const long long max_edges =
        static_cast<long long>(config.vertex_count) * (config.vertex_count - 1) / 2;
    if (static_cast<long long>(config.edge_count) > max_edges) {
        throw std::invalid_argument("edge_count is larger than the number of simple undirected edges");
    }
    if (config.graph_mode != "uniform" &&
        config.graph_mode != "skewed" &&
        config.graph_mode != "clustered") {
        throw std::invalid_argument("unsupported graph_mode: " + config.graph_mode);
    }
    if (config.partition_strategy != "vertex_range" &&
        config.partition_strategy != "edge_hash" &&
        config.partition_strategy != "degree_balance") {
        throw std::invalid_argument("unsupported partition_strategy: " + config.partition_strategy);
    }
    if (config.top_k <= 0) {
        throw std::invalid_argument("top_k must be positive");
    }
}

}  // namespace hccl_csr_graph


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/src/csr_graph.cpp
#include "csr_graph.hpp"

#include <algorithm>
#include <cmath>
#include <cstdint>
#include <numeric>
#include <random>
#include <stdexcept>
#include <unordered_set>
#include <vector>

namespace hccl_csr_graph {
namespace {

int randint(std::mt19937& rng, int low, int high) {
    std::uniform_int_distribution<int> dist(low, high);
    return dist(rng);
}

double random_unit(std::mt19937& rng) {
    std::uniform_real_distribution<double> dist(0.0, 1.0);
    return dist(rng);
}

std::uint64_t edge_key(int u, int v) {
    // 无向边统一保存为 (min, max)，再压入 64 位整数用于快速判重。
    if (u > v) {
        std::swap(u, v);
    }
    return (static_cast<std::uint64_t>(static_cast<std::uint32_t>(u)) << 32U) |
           static_cast<std::uint32_t>(v);
}

int choose_from_candidates(
    std::mt19937& rng,
    int vertex_count,
    const std::vector<int>& candidates,
    double candidate_ratio) {
    if (!candidates.empty() && random_unit(rng) < candidate_ratio) {
        return candidates[static_cast<std::size_t>(
            randint(rng, 0, static_cast<int>(candidates.size()) - 1))];
    }
    return randint(rng, 0, vertex_count - 1);
}

void add_edge_if_new(
    std::vector<Edge>& edges,
    std::unordered_set<std::uint64_t>& seen,
    int u,
    int v) {
    // 实验生成简单无向图：拒绝自环，并利用 seen 去除重复边。
    if (u == v) {
        return;
    }
    if (u > v) {
        std::swap(u, v);
    }
    const std::uint64_t key = edge_key(u, v);
    if (seen.insert(key).second) {
        edges.push_back(Edge{u, v});
    }
}

std::vector<int> make_range(int start, int end) {
    std::vector<int> values;
    values.reserve(static_cast<std::size_t>(std::max(0, end - start)));
    for (int value = start; value < end; ++value) {
        values.push_back(value);
    }
    return values;
}

CsrGraph build_csr(int vertex_count, std::vector<Edge> edges) {
    CsrGraph graph;
    graph.vertex_count = vertex_count;
    graph.edges = std::move(edges);

    // 第一次扫描边表得到每个顶点的度数，也是 CSR 每行所需空间。
    std::vector<int> degrees(static_cast<std::size_t>(vertex_count), 0);
    for (const auto& edge : graph.edges) {
        degrees[static_cast<std::size_t>(edge.src)] += 1;
        degrees[static_cast<std::size_t>(edge.dst)] += 1;
    }

    // 度数前缀和生成 row_ptr，顶点 v 的邻接区间为 [row_ptr[v], row_ptr[v+1])。
    graph.row_ptr.assign(static_cast<std::size_t>(vertex_count) + 1, 0);
    for (int vertex = 0; vertex < vertex_count; ++vertex) {
        graph.row_ptr[static_cast<std::size_t>(vertex + 1)] =
            graph.row_ptr[static_cast<std::size_t>(vertex)] +
            degrees[static_cast<std::size_t>(vertex)];
    }

    // 无向边写入两个方向，因此 col_indices 长度等于 2 * edge_count。
    graph.col_indices.assign(static_cast<std::size_t>(graph.row_ptr.back()), 0);
    std::vector<int> cursor = graph.row_ptr;
    for (const auto& edge : graph.edges) {
        graph.col_indices[static_cast<std::size_t>(cursor[static_cast<std::size_t>(edge.src)]++)] = edge.dst;
        graph.col_indices[static_cast<std::size_t>(cursor[static_cast<std::size_t>(edge.dst)]++)] = edge.src;
    }

    // 每行邻接顶点排序，保证 CSR 内容稳定且便于观察。
    for (int vertex = 0; vertex < vertex_count; ++vertex) {
        const int start = graph.row_ptr[static_cast<std::size_t>(vertex)];
        const int end = graph.row_ptr[static_cast<std::size_t>(vertex + 1)];
        std::sort(
            graph.col_indices.begin() + static_cast<std::ptrdiff_t>(start),
            graph.col_indices.begin() + static_cast<std::ptrdiff_t>(end));
    }

    return graph;
}

}  // namespace

int CsrGraph::edge_count() const {
    return static_cast<int>(edges.size());
}

int CsrGraph::adjacency_count() const {
    return static_cast<int>(col_indices.size());
}

int CsrGraph::degree(int vertex) const {
    if (vertex < 0 || vertex >= vertex_count) {
        throw std::out_of_range("vertex id out of range");
    }
    // CSR 中一行的长度就是该顶点的度数。
    return row_ptr[static_cast<std::size_t>(vertex + 1)] -
           row_ptr[static_cast<std::size_t>(vertex)];
}

CsrGraph generate_graph(const ExperimentConfig& config) {
    validate_config(config);

    std::mt19937 rng(config.seed);
    std::vector<Edge> edges;
    edges.reserve(static_cast<std::size_t>(config.edge_count));
    std::unordered_set<std::uint64_t> seen;
    seen.reserve(static_cast<std::size_t>(config.edge_count) * 2 + 1);

    // skewed 模式把前一小部分顶点作为热点候选。
    const int hot_count = std::max(4, config.vertex_count / 32);
    std::vector<int> hot_vertices(static_cast<std::size_t>(hot_count));
    std::iota(hot_vertices.begin(), hot_vertices.end(), 0);

    // clustered 模式把顶点空间划成若干簇，并提高簇内连边概率。
    const int cluster_count =
        std::max(4, std::min(16, static_cast<int>(std::sqrt(config.vertex_count))));
    const int cluster_size =
        std::max(1, static_cast<int>(std::ceil(static_cast<double>(config.vertex_count) / cluster_count)));

    long long attempts = 0;
    const long long max_attempts = std::max<long long>(1000, static_cast<long long>(config.edge_count) * 200);

    while (static_cast<int>(edges.size()) < config.edge_count && attempts < max_attempts) {
        ++attempts;
        int u = 0;
        int v = 0;

        if (config.graph_mode == "uniform") {
            // 两个端点在全体顶点中独立均匀采样。
            u = randint(rng, 0, config.vertex_count - 1);
            v = randint(rng, 0, config.vertex_count - 1);
        } else if (config.graph_mode == "skewed") {
            // 端点以较高概率从热点顶点中采样，形成高度数服务节点。
            u = choose_from_candidates(rng, config.vertex_count, hot_vertices, 0.74);
            v = choose_from_candidates(rng, config.vertex_count, hot_vertices, 0.36);
        } else if (config.graph_mode == "clustered") {
            // 两个端点优先选自同一簇，模拟模块化依赖关系。
            const int cluster_id =
                random_unit(rng) < 0.72 ? randint(rng, 0, cluster_count - 1)
                                         : static_cast<int>(edges.size()) % cluster_count;
            const int start = cluster_id * cluster_size;
            const int end = std::min(config.vertex_count, start + cluster_size);
            const std::vector<int> cluster_vertices = make_range(start, end);
            u = choose_from_candidates(rng, config.vertex_count, cluster_vertices, 0.88);
            v = choose_from_candidates(rng, config.vertex_count, cluster_vertices, 0.88);
        }

        add_edge_if_new(edges, seen, u, v);
    }

    if (static_cast<int>(edges.size()) != config.edge_count) {
        throw std::runtime_error("failed to generate requested number of unique graph edges");
    }

    // 固定边顺序，保证 edge_hash 分片和多次实验可复现。
    std::sort(edges.begin(), edges.end(), [](const Edge& lhs, const Edge& rhs) {
        if (lhs.src != rhs.src) {
            return lhs.src < rhs.src;
        }
        return lhs.dst < rhs.dst;
    });

    return build_csr(config.vertex_count, std::move(edges));
}

}  // namespace hccl_csr_graph


### 4.2 图分片与局部度数统计

`partition_graph`实现三种分片策略，并统一计算每个Rank的分片单元数量和邻接项数量。`compute_local_degree`根据当前Rank获得的顶点或边分片生成全局长度的局部度数贡献向量。


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/src/partition.cpp
#include "partition.hpp"

#include <algorithm>
#include <cmath>
#include <cstdint>
#include <functional>
#include <queue>
#include <stdexcept>

namespace hccl_csr_graph {
namespace {

void finalize_counts(const CsrGraph& graph, PartitionResult& result) {
    // 统一计算每个 Rank 的分片单元数和实际邻接项处理量。
    const int world_size = static_cast<int>(result.vertex_ids_by_rank.size());
    result.item_counts.clear();
    result.adjacency_counts.clear();
    result.item_counts.reserve(static_cast<std::size_t>(world_size));
    result.adjacency_counts.reserve(static_cast<std::size_t>(world_size));

    for (int rank = 0; rank < world_size; ++rank) {
        int adjacency_count = 0;
        if (result.edge_based) {
            const auto& edge_ids = result.edge_ids_by_rank[static_cast<std::size_t>(rank)];
            result.item_counts.push_back(static_cast<int>(edge_ids.size()));
            // 每条无向边对两个端点各贡献一次度数。
            adjacency_count = static_cast<int>(edge_ids.size()) * 2;
        } else {
            const auto& vertices = result.vertex_ids_by_rank[static_cast<std::size_t>(rank)];
            result.item_counts.push_back(static_cast<int>(vertices.size()));
            for (int vertex : vertices) {
                adjacency_count += graph.degree(vertex);
            }
        }
        result.adjacency_counts.push_back(adjacency_count);
    }
}

}  // namespace

PartitionResult partition_graph(const CsrGraph& graph, int world_size, const std::string& strategy) {
    if (world_size <= 0) {
        throw std::invalid_argument("world_size must be positive");
    }

    PartitionResult result;
    result.strategy = strategy;
    result.vertex_ids_by_rank.resize(static_cast<std::size_t>(world_size));
    result.edge_ids_by_rank.resize(static_cast<std::size_t>(world_size));

    if (strategy == "vertex_range") {
        // 按顶点编号切连续区间，顶点数接近均衡，但邻接量可能严重不均。
        const int chunk = static_cast<int>(std::ceil(static_cast<double>(graph.vertex_count) / world_size));
        for (int rank = 0; rank < world_size; ++rank) {
            const int start = rank * chunk;
            const int end = std::min(graph.vertex_count, start + chunk);
            for (int vertex = start; vertex < end; ++vertex) {
                result.vertex_ids_by_rank[static_cast<std::size_t>(rank)].push_back(vertex);
            }
        }
    } else if (strategy == "edge_hash") {
        // 对边编号做乘法哈希，每条边只属于一个 Rank。
        result.edge_based = true;
        for (int edge_id = 0; edge_id < graph.edge_count(); ++edge_id) {
            const std::uint32_t hashed = static_cast<std::uint32_t>(edge_id) * 2654435761U;
            const int rank = static_cast<int>(hashed % static_cast<std::uint32_t>(world_size));
            result.edge_ids_by_rank[static_cast<std::size_t>(rank)].push_back(edge_id);
        }
    } else if (strategy == "degree_balance") {
        // LPT 贪心：按度数从大到小，把顶点放到当前邻接负载最小的 Rank。
        using HeapItem = std::pair<int, int>;  // current adjacency entries, rank
        std::priority_queue<HeapItem, std::vector<HeapItem>, std::greater<HeapItem>> heap;
        for (int rank = 0; rank < world_size; ++rank) {
            heap.push({0, rank});
        }

        std::vector<int> vertices(static_cast<std::size_t>(graph.vertex_count));
        for (int vertex = 0; vertex < graph.vertex_count; ++vertex) {
            vertices[static_cast<std::size_t>(vertex)] = vertex;
        }
        std::sort(vertices.begin(), vertices.end(), [&](int lhs, int rhs) {
            const int lhs_degree = graph.degree(lhs);
            const int rhs_degree = graph.degree(rhs);
            if (lhs_degree != rhs_degree) {
                return lhs_degree > rhs_degree;
            }
            return lhs < rhs;
        });

        // 小根堆顶部始终是当前累计邻接量最小的 Rank。
        for (int vertex : vertices) {
            const auto [current_adjacency, rank] = heap.top();
            heap.pop();
            result.vertex_ids_by_rank[static_cast<std::size_t>(rank)].push_back(vertex);
            heap.push({current_adjacency + graph.degree(vertex), rank});
        }
        for (auto& vertices_for_rank : result.vertex_ids_by_rank) {
            std::sort(vertices_for_rank.begin(), vertices_for_rank.end());
        }
    } else {
        throw std::invalid_argument("unsupported partition_strategy: " + strategy);
    }

    finalize_counts(graph, result);
    return result;
}

}  // namespace hccl_csr_graph


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/src/degree.cpp
#include "degree.hpp"

#include <algorithm>
#include <stdexcept>

namespace hccl_csr_graph {

PaddingInfo compute_padding(std::size_t vector_size, int world_size) {
    if (world_size <= 0) {
        throw std::invalid_argument("world_size must be positive");
    }
    const std::size_t ranks = static_cast<std::size_t>(world_size);
    // 向上取整分片长度，再把总长度补齐到 world_size 的整数倍。
    const std::size_t shard_size = (vector_size + ranks - 1) / ranks;
    return PaddingInfo{shard_size * ranks, shard_size};
}

CountVector compute_local_degree(const CsrGraph& graph, const PartitionResult& partition, int rank) {
    if (rank < 0 || rank >= static_cast<int>(partition.vertex_ids_by_rank.size())) {
        throw std::invalid_argument("rank out of range");
    }

    // 每个 Rank 都生成与全局顶点数等长的稀疏贡献向量，未负责位置保持 0。
    CountVector degree(static_cast<std::size_t>(graph.vertex_count), 0);

    if (partition.edge_based) {
        // 边分片中每条无向边分别给 src 和 dst 的局部度数加 1。
        for (int edge_id : partition.edge_ids_by_rank[static_cast<std::size_t>(rank)]) {
            const Edge& edge = graph.edges[static_cast<std::size_t>(edge_id)];
            degree[static_cast<std::size_t>(edge.src)] += 1;
            degree[static_cast<std::size_t>(edge.dst)] += 1;
        }
        return degree;
    }

    // 顶点分片拥有完整 CSR 行，因此可直接写入该顶点的完整度数。
    for (int vertex : partition.vertex_ids_by_rank[static_cast<std::size_t>(rank)]) {
        degree[static_cast<std::size_t>(vertex)] = graph.degree(vertex);
    }
    return degree;
}

CountVector serial_reference_degree(const CsrGraph& graph) {
    // 串行扫描所有 CSR 行，作为集合通信结果的标准答案。
    CountVector degree(static_cast<std::size_t>(graph.vertex_count), 0);
    for (int vertex = 0; vertex < graph.vertex_count; ++vertex) {
        degree[static_cast<std::size_t>(vertex)] = graph.degree(vertex);
    }
    return degree;
}

CountVector pad_vector(const CountVector& values, std::size_t padded_size) {
    if (values.size() > padded_size) {
        throw std::invalid_argument("padded_size cannot be smaller than vector size");
    }
    // 补齐位置填 0，不会改变真实顶点的规约结果。
    CountVector padded = values;
    padded.resize(padded_size, 0);
    return padded;
}

CountVector unpad_vector(const CountVector& values, std::size_t original_size) {
    if (values.size() < original_size) {
        throw std::invalid_argument("original_size cannot be larger than vector size");
    }
    return CountVector(values.begin(), values.begin() + static_cast<std::ptrdiff_t>(original_size));
}

std::pair<bool, std::vector<std::size_t>> verify_result(
    const CountVector& global_degree,
    const CountVector& reference_degree) {
    const std::size_t compare_size = std::min(global_degree.size(), reference_degree.size());
    std::vector<std::size_t> mismatches;

    // 逐顶点比较通信恢复度数和串行参考度数。
    for (std::size_t index = 0; index < compare_size; ++index) {
        if (global_degree[index] != reference_degree[index]) {
            mismatches.push_back(index);
        }
    }
    for (std::size_t index = compare_size; index < global_degree.size(); ++index) {
        mismatches.push_back(index);
    }
    for (std::size_t index = compare_size; index < reference_degree.size(); ++index) {
        mismatches.push_back(index);
    }
    return {mismatches.empty(), mismatches};
}

BufferEstimate estimate_buffers(
    std::size_t padded_vertex_count,
    std::size_t shard_size,
    int world_size,
    const CsrGraph& graph) {
    // Count 为 int64，CSR 与边表则分别按 int 和 Edge 的实际大小估算。
    const std::size_t element_size = sizeof(Count);
    return BufferEstimate{
        padded_vertex_count * element_size,
        shard_size * element_size,
        padded_vertex_count * element_size,
        padded_vertex_count * element_size * static_cast<std::size_t>(world_size),
        graph.row_ptr.size() * sizeof(int),
        graph.col_indices.size() * sizeof(int),
        graph.edges.size() * sizeof(Edge),
    };
}

}  // namespace hccl_csr_graph


### 4.3 CPU模拟集合通信

模拟后端先将每个Rank的向量补齐到相同长度，逐位置求和并按连续区间切分，随后拼接分片得到完整向量。该实现保留与真实HCCL后端相同的`CommunicationResult`结构，便于复用校验、缓冲区估算和结果报告。


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/src/simulate_collectives.cpp
#include "collectives.hpp"

#include <chrono>
#include <stdexcept>

namespace hccl_csr_graph {

CommunicationResult run_simulated_collectives(
    const std::vector<CountVector>& local_degrees,
    int world_size) {
    if (world_size <= 0) {
        throw std::invalid_argument("world_size must be positive");
    }
    if (local_degrees.empty()) {
        throw std::invalid_argument("local_degrees cannot be empty");
    }
    if (local_degrees.size() != static_cast<std::size_t>(world_size)) {
        throw std::invalid_argument("local_degrees size must equal world_size");
    }

    const std::size_t vertex_count = local_degrees.front().size();
    for (const auto& degree : local_degrees) {
        if (degree.size() != vertex_count) {
            throw std::invalid_argument("all local degree vectors must have the same size");
        }
    }

    // 模拟真实集合通信的等长缓冲区要求，先补齐所有局部向量。
    const PaddingInfo padding = compute_padding(vertex_count, world_size);
    std::vector<CountVector> padded_degrees;
    padded_degrees.reserve(local_degrees.size());
    for (const auto& degree : local_degrees) {
        padded_degrees.push_back(pad_vector(degree, padding.padded_size));
    }

    const auto reduce_start = std::chrono::steady_clock::now();
    // Reduce：对所有 Rank 的同一顶点位置求和，得到全局度数。
    CountVector global_sum(padding.padded_size, 0);
    for (const auto& degree : padded_degrees) {
        for (std::size_t index = 0; index < degree.size(); ++index) {
            global_sum[index] += degree[index];
        }
    }

    // Scatter：将全局度数向量切成 world_size 个连续等长分片。
    std::vector<CountVector> shards;
    shards.reserve(static_cast<std::size_t>(world_size));
    for (int rank = 0; rank < world_size; ++rank) {
        const std::size_t start = static_cast<std::size_t>(rank) * padding.shard_size;
        shards.emplace_back(
            global_sum.begin() + static_cast<std::ptrdiff_t>(start),
            global_sum.begin() + static_cast<std::ptrdiff_t>(start + padding.shard_size));
    }
    const auto reduce_end = std::chrono::steady_clock::now();

    const auto gather_start = std::chrono::steady_clock::now();
    // AllGather：按 Rank 顺序拼回完整全局度数向量。
    CountVector gathered;
    gathered.reserve(padding.padded_size);
    for (const auto& shard : shards) {
        gathered.insert(gathered.end(), shard.begin(), shard.end());
    }
    const auto gather_end = std::chrono::steady_clock::now();

    CommunicationResult result;
    result.padded_vertex_count = padding.padded_size;
    result.shard_size = padding.shard_size;
    result.reduced_shards = std::move(shards);
    result.gathered_degree = std::move(gathered);
    result.reduce_scatter_ms =
        std::chrono::duration<double, std::milli>(reduce_end - reduce_start).count();
    result.all_gather_ms =
        std::chrono::duration<double, std::milli>(gather_end - gather_start).count();
    return result;
}

}  // namespace hccl_csr_graph


### 4.4 真实HCCL后端

`run_hccl_collectives`负责初始化Ascend设备、创建RootInfo、建立HCCL通信域、分配设备内存并执行ReduceScatter和AllGather。计时只包围对应集合通信调用，结果复制回主机后交给公共结果流程验证。


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/src/hccl_backend.cpp
#include "hccl_backend.hpp"

#include <chrono>
#include <stdexcept>

#if defined(ENABLE_HCCL)
#include <acl/acl.h>
#include <hccl/hccl.h>

#include <cstdint>
#include <cstring>
#include <fstream>
#include <sstream>
#include <thread>
#endif

namespace hccl_csr_graph {

#if defined(ENABLE_HCCL)
namespace {

void check_acl(aclError result, const char* expression) {
    if (result != ACL_ERROR_NONE) {
        std::ostringstream oss;
        oss << expression << " failed, acl error=" << static_cast<int>(result);
        throw std::runtime_error(oss.str());
    }
}

void check_hccl(HcclResult result, const char* expression) {
    if (result != HCCL_SUCCESS) {
        std::ostringstream oss;
        oss << expression << " failed, hccl error=" << static_cast<int>(result);
        throw std::runtime_error(oss.str());
    }
}

#define CHECK_ACL(expr) check_acl((expr), #expr)
#define CHECK_HCCL(expr) check_hccl((expr), #expr)

void write_root_info(const std::string& path, const HcclRootInfo& root_info) {
    // rank0 生成通信域 RootInfo，并通过临时文件传递给其他 Rank。
    std::ofstream output(path, std::ios::binary | std::ios::trunc);
    if (!output) {
        throw std::runtime_error("failed to write HCCL root info file: " + path);
    }
    output.write(reinterpret_cast<const char*>(&root_info), sizeof(root_info));
}

HcclRootInfo read_root_info(const std::string& path) {
    HcclRootInfo root_info;
    std::memset(&root_info, 0, sizeof(root_info));

    // 非零 Rank 可能先启动，因此轮询等待 rank0 完整写入文件。
    for (int retry = 0; retry < 600; ++retry) {
        std::ifstream input(path, std::ios::binary);
        if (input) {
            input.read(reinterpret_cast<char*>(&root_info), sizeof(root_info));
            if (input.gcount() == static_cast<std::streamsize>(sizeof(root_info))) {
                return root_info;
            }
        }
        std::this_thread::sleep_for(std::chrono::milliseconds(100));
    }

    throw std::runtime_error("timeout while waiting for HCCL root info file: " + path);
}

void* acl_malloc(std::size_t bytes) {
    // HCCL 输入、分片和聚合结果均使用 Ascend 设备内存。
    void* ptr = nullptr;
    CHECK_ACL(aclrtMalloc(&ptr, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
    return ptr;
}

std::vector<CountVector> split_gathered_into_shards(
    const CountVector& gathered,
    std::size_t shard_size,
    int world_size) {
    // 为统一报告格式，把 AllGather 完整结果重新切成各 Rank 的规约分片。
    std::vector<CountVector> shards;
    shards.reserve(static_cast<std::size_t>(world_size));
    for (int rank = 0; rank < world_size; ++rank) {
        const std::size_t start = static_cast<std::size_t>(rank) * shard_size;
        shards.emplace_back(
            gathered.begin() + static_cast<std::ptrdiff_t>(start),
            gathered.begin() + static_cast<std::ptrdiff_t>(start + shard_size));
    }
    return shards;
}

}  // namespace

bool is_hccl_compiled() {
    return true;
}

CommunicationResult run_hccl_collectives(
    const CountVector& local_degree,
    std::size_t vertex_count,
    const HcclRuntimeOptions& options) {
    if (options.world_size < 1) {
        throw std::invalid_argument("backend=hccl requires world_size >= 1");
    }
    if (options.rank < 0 || options.rank >= options.world_size) {
        throw std::invalid_argument("rank must be in [0, world_size)");
    }

    // 顶点数补齐到可等分长度，每个 Rank 输入完整局部贡献向量。
    const PaddingInfo padding = compute_padding(vertex_count, options.world_size);
    const CountVector padded = pad_vector(local_degree, padding.padded_size);
    CountVector host_gathered(padding.padded_size, 0);

    aclrtStream stream = nullptr;
    HcclComm comm = nullptr;
    void* send_device = nullptr;
    void* shard_device = nullptr;
    void* gathered_device = nullptr;

    try {
        // 初始化 ACL、绑定当前 NPU，并创建承载异步 HCCL 操作的流。
        CHECK_ACL(aclInit(nullptr));
        CHECK_ACL(aclrtSetDevice(options.local_device));
        CHECK_ACL(aclrtCreateStream(&stream));

        // 所有 Rank 使用相同 RootInfo 和 world_size，以各自 rank 加入通信域。
        HcclRootInfo root_info;
        if (options.rank == 0) {
            CHECK_HCCL(HcclGetRootInfo(&root_info));
            write_root_info(options.root_info_file, root_info);
        } else {
            root_info = read_root_info(options.root_info_file);
        }

        CHECK_HCCL(HcclCommInitRootInfo(
            static_cast<std::uint32_t>(options.world_size),
            &root_info,
            static_cast<std::uint32_t>(options.rank),
            &comm));

        // send/gathered 是完整向量，shard 只保存 ReduceScatter 后的一个分片。
        const std::size_t full_bytes = padding.padded_size * sizeof(Count);
        const std::size_t shard_bytes = padding.shard_size * sizeof(Count);
        send_device = acl_malloc(full_bytes);
        shard_device = acl_malloc(shard_bytes);
        gathered_device = acl_malloc(full_bytes);

        // 将当前 Rank 的局部度数贡献从主机复制到设备。
        CHECK_ACL(aclrtMemcpy(
            send_device,
            full_bytes,
            padded.data(),
            full_bytes,
            ACL_MEMCPY_HOST_TO_DEVICE));

        const auto reduce_start = std::chrono::steady_clock::now();
        // 逐顶点求和后按 Rank 切分，每个 Rank 接收 shard_size 个 int64。
        CHECK_HCCL(HcclReduceScatter(
            send_device,
            shard_device,
            static_cast<std::uint64_t>(padding.shard_size),
            HCCL_DATA_TYPE_INT64,
            HCCL_REDUCE_SUM,
            comm,
            stream));
        CHECK_ACL(aclrtSynchronizeStream(stream));
        const auto reduce_end = std::chrono::steady_clock::now();

        const auto gather_start = std::chrono::steady_clock::now();
        // 收集所有规约分片，使每个 Rank 都恢复完整全局度数。
        CHECK_HCCL(HcclAllGather(
            shard_device,
            gathered_device,
            static_cast<std::uint64_t>(padding.shard_size),
            HCCL_DATA_TYPE_INT64,
            comm,
            stream));
        CHECK_ACL(aclrtSynchronizeStream(stream));
        const auto gather_end = std::chrono::steady_clock::now();

        // 把全局度数复制回主机，供 Top-K、指标计算和正确性校验使用。
        CHECK_ACL(aclrtMemcpy(
            host_gathered.data(),
            full_bytes,
            gathered_device,
            full_bytes,
            ACL_MEMCPY_DEVICE_TO_HOST));

        CommunicationResult result;
        result.padded_vertex_count = padding.padded_size;
        result.shard_size = padding.shard_size;
        result.reduced_shards = split_gathered_into_shards(host_gathered, padding.shard_size, options.world_size);
        result.gathered_degree = std::move(host_gathered);
        result.reduce_scatter_ms =
            std::chrono::duration<double, std::milli>(reduce_end - reduce_start).count();
        result.all_gather_ms =
            std::chrono::duration<double, std::milli>(gather_end - gather_start).count();

        // 正常路径按通信域、设备缓冲区、流、设备和 ACL 的顺序释放资源。
        CHECK_HCCL(HcclCommDestroy(comm));
        comm = nullptr;
        CHECK_ACL(aclrtFree(send_device));
        send_device = nullptr;
        CHECK_ACL(aclrtFree(shard_device));
        shard_device = nullptr;
        CHECK_ACL(aclrtFree(gathered_device));
        gathered_device = nullptr;
        CHECK_ACL(aclrtDestroyStream(stream));
        stream = nullptr;
        CHECK_ACL(aclrtResetDevice(options.local_device));
        CHECK_ACL(aclFinalize());
        return result;
    } catch (...) {
        // 任意阶段失败都进行兜底清理，然后把原异常继续抛给 main。
        if (comm != nullptr) {
            HcclCommDestroy(comm);
        }
        if (send_device != nullptr) {
            aclrtFree(send_device);
        }
        if (shard_device != nullptr) {
            aclrtFree(shard_device);
        }
        if (gathered_device != nullptr) {
            aclrtFree(gathered_device);
        }
        if (stream != nullptr) {
            aclrtDestroyStream(stream);
        }
        aclrtResetDevice(options.local_device);
        aclFinalize();
        throw;
    }
}

#else

bool is_hccl_compiled() {
    return false;
}

CommunicationResult run_hccl_collectives(
    const CountVector&,
    std::size_t,
    const HcclRuntimeOptions&) {
    throw std::runtime_error(
        "HCCL backend is not compiled. Rebuild with cmake -DENABLE_HCCL=ON.");
}

#endif

}  // namespace hccl_csr_graph


### 4.5 负载指标与结果报告

`compute_partition_metrics`同时统计分片单元和邻接项两类负载；`topk_vertices`返回全局或局部最高度数顶点。`print_results`输出完整表格，`write_result_summary`写入便于Notebook和实验报告读取的键值摘要。


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/src/metrics.cpp
#include "metrics.hpp"

#include <algorithm>
#include <numeric>

namespace hccl_csr_graph {
namespace {

double average(const std::vector<int>& values) {
    if (values.empty()) {
        return 0.0;
    }
    const int sum = std::accumulate(values.begin(), values.end(), 0);
    return static_cast<double>(sum) / static_cast<double>(values.size());
}

double safe_ratio(int max_value, double avg_value) {
    return avg_value > 0.0 ? static_cast<double>(max_value) / avg_value : 0.0;
}

}  // namespace

PartitionMetrics compute_partition_metrics(const PartitionResult& partition) {
    PartitionMetrics metrics;
    if (partition.item_counts.empty() || partition.adjacency_counts.empty()) {
        return metrics;
    }

    // item 指顶点或边；adjacency 更接近实际遍历和度数计算工作量。
    metrics.min_items = *std::min_element(partition.item_counts.begin(), partition.item_counts.end());
    metrics.max_items = *std::max_element(partition.item_counts.begin(), partition.item_counts.end());
    metrics.avg_items = average(partition.item_counts);
    metrics.item_imbalance = safe_ratio(metrics.max_items, metrics.avg_items);

    metrics.min_adjacency = *std::min_element(partition.adjacency_counts.begin(), partition.adjacency_counts.end());
    metrics.max_adjacency = *std::max_element(partition.adjacency_counts.begin(), partition.adjacency_counts.end());
    metrics.avg_adjacency = average(partition.adjacency_counts);
    metrics.adjacency_imbalance = safe_ratio(metrics.max_adjacency, metrics.avg_adjacency);
    return metrics;
}

std::vector<std::pair<int, Count>> topk_vertices(const CountVector& degree, int top_k) {
    // 将向量下标恢复为顶点编号，再按度数降序、编号升序稳定排序。
    std::vector<std::pair<int, Count>> pairs;
    pairs.reserve(degree.size());
    for (std::size_t index = 0; index < degree.size(); ++index) {
        pairs.push_back({static_cast<int>(index), degree[index]});
    }
    std::sort(pairs.begin(), pairs.end(), [](const auto& lhs, const auto& rhs) {
        if (lhs.second != rhs.second) {
            return lhs.second > rhs.second;
        }
        return lhs.first < rhs.first;
    });
    if (top_k < static_cast<int>(pairs.size())) {
        pairs.resize(static_cast<std::size_t>(top_k));
    }
    return pairs;
}

}  // namespace hccl_csr_graph


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/src/result_report.cpp
#include "result_report.hpp"

#include <algorithm>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <numeric>
#include <stdexcept>

namespace hccl_csr_graph {
namespace {

std::string pass_fail(bool value) {
    return value ? "PASS" : "FAIL";
}

Count sum_vector(const CountVector& values) {
    return std::accumulate(values.begin(), values.end(), Count{0});
}

int nonzero_count(const CountVector& values) {
    return static_cast<int>(std::count_if(values.begin(), values.end(), [](Count value) {
        return value != 0;
    }));
}

std::string item_label(const PartitionResult& partition) {
    return partition.edge_based ? "edges" : "vertices";
}

}  // namespace

void print_results(const ExperimentResult& result) {
    // 输出顺序与实验流程一致：配置、CSR、策略、Rank 负载、通信、Top-K 和校验。
    const auto partition_metrics = compute_partition_metrics(result.partition);

    std::cout << "Experiment configuration\n";
    std::cout << std::string(88, '-') << '\n';
    std::cout << std::left << std::setw(30) << "backend" << ": " << backend_name(result.config.backend) << '\n';
    std::cout << std::left << std::setw(30) << "world_size" << ": " << result.config.world_size << '\n';
    if (result.config.backend == Backend::Hccl) {
        std::cout << std::left << std::setw(30) << "rank" << ": " << result.config.rank << '\n';
        std::cout << std::left << std::setw(30) << "local_device" << ": " << result.config.local_device << '\n';
    }
    std::cout << std::left << std::setw(30) << "vertex_count" << ": " << result.graph.vertex_count << '\n';
    std::cout << std::left << std::setw(30) << "edge_count" << ": " << result.graph.edge_count() << '\n';
    std::cout << std::left << std::setw(30) << "adjacency_count" << ": " << result.graph.adjacency_count() << '\n';
    std::cout << std::left << std::setw(30) << "graph_mode" << ": " << result.config.graph_mode << '\n';
    std::cout << std::left << std::setw(30) << "partition_strategy" << ": " << result.config.partition_strategy << '\n';
    std::cout << std::left << std::setw(30) << "padded_vertex_count" << ": " << result.communication.padded_vertex_count << '\n';
    std::cout << std::left << std::setw(30) << "shard_size" << ": " << result.communication.shard_size << '\n';
    std::cout << std::left << std::setw(30) << "seed" << ": " << result.config.seed << '\n';
    std::cout << std::string(88, '-') << '\n';

    // CSR 表展示压缩数组规模以及图整体度数特征。
    std::cout << "\nCSR graph storage\n";
    std::cout << std::string(88, '-') << '\n';
    const auto reference_top = topk_vertices(result.reference_degree, 1);
    const Count max_degree = reference_top.empty() ? 0 : reference_top.front().second;
    const double avg_degree = result.graph.vertex_count > 0
                                  ? static_cast<double>(result.graph.adjacency_count()) / result.graph.vertex_count
                                  : 0.0;
    std::cout << std::left << std::setw(30) << "row_ptr_length" << ": " << result.graph.row_ptr.size() << '\n';
    std::cout << std::left << std::setw(30) << "col_indices_length" << ": " << result.graph.col_indices.size() << '\n';
    std::cout << std::left << std::setw(30) << "avg_degree" << ": "
              << std::fixed << std::setprecision(3) << avg_degree << '\n';
    std::cout << std::left << std::setw(30) << "max_degree_observed" << ": " << max_degree << '\n';
    std::cout << std::string(88, '-') << '\n';

    // 同时比较单元数量均衡和更接近计算量的邻接项均衡。
    std::cout << "\nPartition strategy comparison\n";
    std::cout << std::string(88, '-') << '\n';
    std::cout << std::left << std::setw(16) << "strategy"
              << std::setw(10) << "unit"
              << std::right << std::setw(10) << "min_items"
              << std::setw(11) << "max_items"
              << std::setw(12) << "item_imb"
              << std::setw(10) << "min_adj"
              << std::setw(10) << "max_adj"
              << std::setw(10) << "adj_imb" << '\n';
    std::cout << std::string(88, '-') << '\n';
    for (const auto& strategy : result.strategy_results) {
        const auto metrics = compute_partition_metrics(strategy);
        std::cout << std::left << std::setw(16) << strategy.strategy
                  << std::setw(10) << item_label(strategy)
                  << std::right << std::setw(10) << metrics.min_items
                  << std::setw(11) << metrics.max_items
                  << std::setw(12) << std::fixed << std::setprecision(3) << metrics.item_imbalance
                  << std::setw(10) << metrics.min_adjacency
                  << std::setw(10) << metrics.max_adjacency
                  << std::setw(10) << std::fixed << std::setprecision(3) << metrics.adjacency_imbalance << '\n';
    }
    std::cout << std::string(88, '-') << '\n';

    // 展开当前选中策略在每个 Rank 上的实际分片负载和局部热点顶点。
    std::cout << "\nSelected partition load per rank\n";
    std::cout << std::string(88, '-') << '\n';
    std::cout << std::right << std::setw(4) << "rank"
              << std::setw(10) << item_label(result.partition)
              << std::setw(12) << "adjacency"
              << std::setw(12) << "item_ratio"
              << std::setw(11) << "adj_ratio"
              << "  top_local_vertices\n";
    std::cout << std::string(88, '-') << '\n';
    for (std::size_t rank = 0; rank < result.partition.item_counts.size(); ++rank) {
        const auto top_local = topk_vertices(result.local_degrees[rank], 3);
        std::cout << std::right << std::setw(4) << rank
                  << std::setw(10) << result.partition.item_counts[rank]
                  << std::setw(12) << result.partition.adjacency_counts[rank]
                  << std::setw(12) << std::fixed << std::setprecision(3)
                  << result.partition.item_counts[rank] / partition_metrics.avg_items
                  << std::setw(11) << std::fixed << std::setprecision(3)
                  << result.partition.adjacency_counts[rank] / partition_metrics.avg_adjacency
                  << "  ";
        for (std::size_t i = 0; i < top_local.size(); ++i) {
            if (i > 0) {
                std::cout << ", ";
            }
            std::cout << 'v' << top_local[i].first << ':' << top_local[i].second;
        }
        std::cout << '\n';
    }
    std::cout << std::string(88, '-') << '\n';

    // shard_sum 之和应等于全部顶点度数之和，即无向边数的两倍。
    std::cout << "\nReduceScatter shard summary\n";
    std::cout << std::string(72, '-') << '\n';
    std::cout << std::right << std::setw(4) << "rank"
              << std::setw(12) << "shard_len"
              << std::setw(14) << "shard_sum"
              << std::setw(12) << "nonzero" << '\n';
    std::cout << std::string(72, '-') << '\n';
    for (std::size_t rank = 0; rank < result.communication.reduced_shards.size(); ++rank) {
        const auto& shard = result.communication.reduced_shards[rank];
        std::cout << std::right << std::setw(4) << rank
                  << std::setw(12) << shard.size()
                  << std::setw(14) << sum_vector(shard)
                  << std::setw(12) << nonzero_count(shard) << '\n';
    }
    std::cout << std::string(72, '-') << '\n';

    std::cout << "\nGlobal top-k vertices\n";
    std::cout << std::string(48, '-') << '\n';
    for (const auto& [vertex, degree] : topk_vertices(result.global_degree, result.config.top_k)) {
        std::cout << 'v' << std::left << std::setw(8) << vertex
                  << std::right << std::setw(10) << degree << '\n';
    }
    std::cout << std::string(48, '-') << '\n';

    std::cout << "\nCommunication time in " << backend_name(result.config.backend) << " backend\n";
    std::cout << std::string(88, '-') << '\n';
    std::cout << std::left << std::setw(30) << "reduce_scatter_ms" << ": "
              << std::fixed << std::setprecision(4) << result.communication.reduce_scatter_ms << '\n';
    std::cout << std::left << std::setw(30) << "all_gather_ms" << ": "
              << std::fixed << std::setprecision(4) << result.communication.all_gather_ms << '\n';
    std::cout << std::string(88, '-') << '\n';

    std::cout << "\nEstimated buffer size, int64\n";
    std::cout << std::string(88, '-') << '\n';
    std::cout << std::left << std::setw(42) << "input_degree_vector_bytes_per_rank"
              << ": " << std::right << std::setw(8) << result.buffers.input_degree_vector_bytes_per_rank << " bytes\n";
    std::cout << std::left << std::setw(42) << "reduce_scatter_shard_bytes_per_rank"
              << ": " << std::right << std::setw(8) << result.buffers.reduce_scatter_shard_bytes_per_rank << " bytes\n";
    std::cout << std::left << std::setw(42) << "all_gather_result_bytes_per_rank"
              << ": " << std::right << std::setw(8) << result.buffers.all_gather_result_bytes_per_rank << " bytes\n";
    std::cout << std::left << std::setw(42) << "all_ranks_input_bytes_total"
              << ": " << std::right << std::setw(8) << result.buffers.all_ranks_input_bytes_total << " bytes\n";
    std::cout << std::left << std::setw(42) << "csr_row_ptr_bytes"
              << ": " << std::right << std::setw(8) << result.buffers.csr_row_ptr_bytes << " bytes\n";
    std::cout << std::left << std::setw(42) << "csr_col_indices_bytes"
              << ": " << std::right << std::setw(8) << result.buffers.csr_col_indices_bytes << " bytes\n";
    std::cout << std::left << std::setw(42) << "edge_list_bytes"
              << ": " << std::right << std::setw(8) << result.buffers.edge_list_bytes << " bytes\n";
    std::cout << std::string(88, '-') << '\n';

    std::cout << "\nCorrectness check\n";
    std::cout << std::string(88, '-') << '\n';
    std::cout << std::left << std::setw(30) << "result" << ": " << pass_fail(result.correct) << '\n';
    std::cout << std::left << std::setw(30) << "mismatch_count" << ": " << result.mismatches.size() << '\n';
    std::cout << std::left << std::setw(30) << "item_imbalance" << ": "
              << std::fixed << std::setprecision(3) << partition_metrics.item_imbalance << '\n';
    std::cout << std::left << std::setw(30) << "adjacency_imbalance" << ": "
              << std::fixed << std::setprecision(3) << partition_metrics.adjacency_imbalance << '\n';
    std::cout << std::string(88, '-') << '\n';
}

void write_result_summary(const std::string& output_path, const ExperimentResult& result) {
    // 精简文本结果供 Notebook、实验报告和后续脚本复用。
    std::ofstream output(output_path);
    if (!output) {
        throw std::runtime_error("failed to open result file: " + output_path);
    }

    const auto metrics = compute_partition_metrics(result.partition);
    output << "backend: " << backend_name(result.config.backend) << '\n';
    output << "world_size: " << result.config.world_size << '\n';
    output << "vertex_count: " << result.graph.vertex_count << '\n';
    output << "edge_count: " << result.graph.edge_count() << '\n';
    output << "adjacency_count: " << result.graph.adjacency_count() << '\n';
    output << "graph_mode: " << result.config.graph_mode << '\n';
    output << "partition_strategy: " << result.config.partition_strategy << '\n';
    output << "padded_vertex_count: " << result.communication.padded_vertex_count << '\n';
    output << "shard_size: " << result.communication.shard_size << '\n';
    output << "reduce_scatter_ms: " << result.communication.reduce_scatter_ms << '\n';
    output << "all_gather_ms: " << result.communication.all_gather_ms << '\n';
    output << "result: " << pass_fail(result.correct) << '\n';
    output << "mismatch_count: " << result.mismatches.size() << '\n';
    output << "item_imbalance: " << metrics.item_imbalance << '\n';
    output << "adjacency_imbalance: " << metrics.adjacency_imbalance << '\n';
    output << "top_vertices:\n";
    for (const auto& [vertex, degree] : topk_vertices(result.global_degree, result.config.top_k)) {
        output << "  v" << vertex << ": " << degree << '\n';
    }
}

}  // namespace hccl_csr_graph


### 4.6 主程序与后端分派

主程序从命令行和环境变量解析参数。`simulate`模式在单进程中构造全部Rank的局部结果；`hccl`模式由每个Rank独立生成相同图，只通信当前Rank的向量，并由Rank 0输出最终汇总。两种后端共享串行参考校验和指标统计流程。


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/src/main.cpp
#include <cstdlib>
#include <exception>
#include <iostream>
#include <stdexcept>
#include <string>
#include <utility>
#include <vector>

#include "collectives.hpp"
#include "csr_graph.hpp"
#include "degree.hpp"
#include "experiment_config.hpp"
#include "hccl_backend.hpp"
#include "partition.hpp"
#include "result_report.hpp"

namespace hccl_csr_graph {
namespace {

int read_env_int(const char* name, int fallback) {
#if defined(_MSC_VER)
    char* value = nullptr;
    std::size_t size = 0;
    if (_dupenv_s(&value, &size, name) != 0 || value == nullptr) {
        return fallback;
    }
    const std::string text(value);
    std::free(value);
    if (text.empty()) {
        return fallback;
    }
    return std::stoi(text);
#else
    const char* value = std::getenv(name);
    if (value == nullptr || std::string(value).empty()) {
        return fallback;
    }
    return std::stoi(value);
#endif
}

void print_usage(const char* program) {
    std::cout
        << "Usage: " << program << " [options]\n"
        << "\n"
        << "Options:\n"
        << "  --backend simulate|hccl          Backend to run, default: simulate\n"
        << "  --world-size N                   Number of logical ranks, default: 4 or WORLD_SIZE\n"
        << "  --rank N                         Current rank for backend=hccl, default: RANK\n"
        << "  --local-device N                 Ascend device id for backend=hccl, default: LOCAL_RANK\n"
        << "  --vertex-count N                 Number of graph vertices\n"
        << "  --edge-count N                   Number of undirected graph edges\n"
        << "  --graph-mode MODE                uniform, skewed, or clustered\n"
        << "  --partition-strategy STRATEGY    vertex_range, edge_hash, or degree_balance\n"
        << "  --seed N                         Deterministic random seed\n"
        << "  --top-k N                        Number of global vertices to print\n"
        << "  --root-info-file PATH            File used to exchange HCCL root info\n"
        << "  --output PATH                    Write a compact result summary\n"
        << "  --help                           Show this message\n";
}

struct ParsedArgs {
    ExperimentConfig config;
    std::string output_path;
};

ParsedArgs parse_args(int argc, char** argv) {
    ParsedArgs parsed;
    // HCCL 启动脚本为每个进程设置 WORLD_SIZE、RANK 和 LOCAL_RANK。
    parsed.config.world_size = read_env_int("WORLD_SIZE", parsed.config.world_size);
    parsed.config.rank = read_env_int("RANK", parsed.config.rank);
    parsed.config.local_device = read_env_int("LOCAL_RANK", parsed.config.rank);

    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto require_value = [&](const std::string& option) -> std::string {
            if (i + 1 >= argc) {
                throw std::invalid_argument(option + " requires a value");
            }
            return argv[++i];
        };

        if (arg == "--help" || arg == "-h") {
            print_usage(argv[0]);
            std::exit(0);
        } else if (arg == "--backend") {
            parsed.config.backend = parse_backend(require_value(arg));
        } else if (arg == "--world-size") {
            parsed.config.world_size = std::stoi(require_value(arg));
        } else if (arg == "--rank") {
            parsed.config.rank = std::stoi(require_value(arg));
        } else if (arg == "--local-device") {
            parsed.config.local_device = std::stoi(require_value(arg));
        } else if (arg == "--vertex-count") {
            parsed.config.vertex_count = std::stoi(require_value(arg));
        } else if (arg == "--edge-count") {
            parsed.config.edge_count = std::stoi(require_value(arg));
        } else if (arg == "--graph-mode") {
            parsed.config.graph_mode = require_value(arg);
        } else if (arg == "--partition-strategy") {
            parsed.config.partition_strategy = require_value(arg);
        } else if (arg == "--seed") {
            parsed.config.seed = static_cast<std::uint32_t>(std::stoul(require_value(arg)));
        } else if (arg == "--top-k") {
            parsed.config.top_k = std::stoi(require_value(arg));
        } else if (arg == "--root-info-file") {
            parsed.config.root_info_file = require_value(arg);
        } else if (arg == "--output") {
            parsed.output_path = require_value(arg);
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }

    if (parsed.output_path.empty()) {
        parsed.output_path = parsed.config.backend == Backend::Simulate
                                 ? "results/simulate_latest.txt"
                                 : "results/hccl_rank0_latest.txt";
    }

    validate_config(parsed.config);
    return parsed;
}

std::vector<CountVector> build_local_degrees(
    const CsrGraph& graph,
    const PartitionResult& partition,
    int world_size) {
    // simulate 模式构造全部 Rank；HCCL 模式也重建全部结果用于 rank0 报表展示。
    std::vector<CountVector> local_degrees;
    local_degrees.reserve(static_cast<std::size_t>(world_size));
    for (int rank = 0; rank < world_size; ++rank) {
        local_degrees.push_back(compute_local_degree(graph, partition, rank));
    }
    return local_degrees;
}

std::vector<PartitionResult> build_strategy_comparison(
    const CsrGraph& graph,
    int world_size) {
    // 同一张图同时运行三种策略，保证负载指标可直接横向比较。
    return {
        partition_graph(graph, world_size, "vertex_range"),
        partition_graph(graph, world_size, "edge_hash"),
        partition_graph(graph, world_size, "degree_balance"),
    };
}

ExperimentResult build_common_result(
    const ExperimentConfig& config,
    const CsrGraph& graph,
    const PartitionResult& partition,
    const std::vector<PartitionResult>& strategy_results,
    const std::vector<CountVector>& local_degrees,
    CommunicationResult communication) {
    // 模拟后端和 HCCL 后端共用相同的去补齐、串行校验、指标与缓冲区估算流程。
    ExperimentResult result;
    result.config = config;
    result.graph = graph;
    result.partition = partition;
    result.strategy_results = strategy_results;
    result.local_degrees = local_degrees;
    result.communication = std::move(communication);
    // AllGather 结果包含补零位置，与真实顶点向量比较前先截断。
    result.global_degree = unpad_vector(
        result.communication.gathered_degree,
        static_cast<std::size_t>(graph.vertex_count));
    result.reference_degree = serial_reference_degree(graph);
    const auto [correct, mismatches] = verify_result(result.global_degree, result.reference_degree);
    result.correct = correct;
    result.mismatches = mismatches;
    result.buffers = estimate_buffers(
        result.communication.padded_vertex_count,
        result.communication.shard_size,
        config.world_size,
        graph);
    return result;
}

int run_simulate(const ExperimentConfig& config, const std::string& output_path) {
    // 单进程依次生成图、分片、局部向量，再模拟两次集合通信。
    const CsrGraph graph = generate_graph(config);
    const PartitionResult partition = partition_graph(graph, config.world_size, config.partition_strategy);
    const std::vector<PartitionResult> strategy_results = build_strategy_comparison(graph, config.world_size);
    const std::vector<CountVector> local_degrees = build_local_degrees(graph, partition, config.world_size);
    CommunicationResult communication = run_simulated_collectives(local_degrees, config.world_size);

    const ExperimentResult result =
        build_common_result(config, graph, partition, strategy_results, local_degrees, std::move(communication));
    print_results(result);
    write_result_summary(output_path, result);
    return result.correct ? 0 : 2;
}

int run_hccl(const ExperimentConfig& config, const std::string& output_path) {
    if (!is_hccl_compiled()) {
        throw std::runtime_error(
            "backend=hccl requires rebuilding with -DENABLE_HCCL=ON and Ascend Toolkit libraries");
    }

    // 所有进程使用相同 seed 独立生成相同图和分片，只通信当前 Rank 的局部向量。

    // 步骤一：生成图数据并构建CSR存储结构。
    const CsrGraph graph = generate_graph(config);

    // 步骤二：执行图分片并分析负载量。
    const PartitionResult partition = partition_graph(graph, config.world_size, config.partition_strategy);
    const std::vector<PartitionResult> strategy_results = build_strategy_comparison(graph, config.world_size);

    // 步骤三：统计本地度数。
    const std::vector<CountVector> local_degrees = build_local_degrees(graph, partition, config.world_size);

    const HcclRuntimeOptions options{
        config.rank,
        config.world_size,
        config.local_device,
        config.root_info_file,
    };

    // 步骤四：执行多进程HCCL集合通信。
    CommunicationResult communication = run_hccl_collectives(
        local_degrees[static_cast<std::size_t>(config.rank)],
        static_cast<std::size_t>(config.vertex_count),
        options);

    // 步骤五：验证并分析实验结果。
    const ExperimentResult result =
        build_common_result(config, graph, partition, strategy_results, local_degrees, std::move(communication));
    // 只让 rank0 打印和写结果，避免多进程日志及文件写入冲突。
    if (config.rank == 0) {
        print_results(result);
        write_result_summary(output_path, result);
    }
    return result.correct ? 0 : 2;
}

}  // namespace
}  // namespace hccl_csr_graph

int main(int argc, char** argv) {
    try {
        // 根据 backend 分派到单进程模拟或真实 HCCL 多进程流程。
        const auto parsed = hccl_csr_graph::parse_args(argc, argv);
        if (parsed.config.backend == hccl_csr_graph::Backend::Simulate) {
            return hccl_csr_graph::run_simulate(parsed.config, parsed.output_path);
        }
        return hccl_csr_graph::run_hccl(parsed.config, parsed.output_path);
    } catch (const std::exception& exc) {
        std::cerr << "error: " << exc.what() << '\n';
        return 1;
    }
}


### 4.7 核心源码检查

检查全部头文件和源文件是否已经写入。全部显示`OK`时，说明Notebook生成的图存储、分片、通信和报告源码完整，可以进入构建和运行阶段。


In [ ]:
required_files = [
    "include/experiment_config.hpp",
    "include/csr_graph.hpp",
    "include/partition.hpp",
    "include/degree.hpp",
    "include/collectives.hpp",
    "include/hccl_backend.hpp",
    "include/metrics.hpp",
    "include/result_report.hpp",
    "src/experiment_config.cpp",
    "src/csr_graph.cpp",
    "src/partition.cpp",
    "src/degree.cpp",
    "src/simulate_collectives.cpp",
    "src/hccl_backend.cpp",
    "src/metrics.cpp",
    "src/result_report.cpp",
    "src/main.cpp",
]

for relative_path in required_files:
    path = WORK_DIR / relative_path
    print(f"{relative_path}: {'OK' if path.exists() else 'MISSING'}")


---
## 5. 结果验证与性能分析

核心源码准备完成后，本节写入CMake构建配置和两个运行脚本。先运行CPU模拟后端检查应用层逻辑，再在已配置CANN且有两张可见NPU的服务器上运行双Rank真实HCCL。Notebook不会伪造HCCL输出，真实通信结果应以服务器生成的日志和摘要为准。


### 5.1 工程构建

`CMakeLists.txt`使用C++17组织9个源文件。默认`ENABLE_HCCL=OFF`时构建模拟后端；设置`ENABLE_HCCL=ON`并提供`ASCEND_TOOLKIT_HOME`时，查找HCCL和AscendCL库并编译真实后端。


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

project(hccl_csr_graph_partition_experiment LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

option(ENABLE_HCCL "Build with Ascend HCCL backend" OFF)
set(ASCEND_TOOLKIT_HOME "" CACHE PATH "Ascend toolkit root, for example /usr/local/Ascend/ascend-toolkit/latest")

add_executable(hccl_csr_graph_partition
    src/main.cpp
    src/experiment_config.cpp
    src/csr_graph.cpp
    src/partition.cpp
    src/degree.cpp
    src/simulate_collectives.cpp
    src/hccl_backend.cpp
    src/metrics.cpp
    src/result_report.cpp
)

target_include_directories(hccl_csr_graph_partition PRIVATE include)

if (MSVC)
    target_compile_options(hccl_csr_graph_partition PRIVATE /W4 /utf-8)
else()
    target_compile_options(hccl_csr_graph_partition PRIVATE -Wall -Wextra -pedantic)
endif()

if (ENABLE_HCCL)
    target_compile_definitions(hccl_csr_graph_partition PRIVATE ENABLE_HCCL=1)

    if (NOT ASCEND_TOOLKIT_HOME)
        if (DEFINED ENV{ASCEND_TOOLKIT_HOME})
            set(ASCEND_TOOLKIT_HOME "$ENV{ASCEND_TOOLKIT_HOME}")
        elseif (EXISTS "/usr/local/Ascend/ascend-toolkit/latest")
            set(ASCEND_TOOLKIT_HOME "/usr/local/Ascend/ascend-toolkit/latest")
        else()
            message(FATAL_ERROR "ENABLE_HCCL=ON requires ASCEND_TOOLKIT_HOME")
        endif()
    endif()

    # CANN headers are third-party headers and may use compiler extensions.
    # Keep strict warnings enabled for the experiment sources themselves.
    target_include_directories(hccl_csr_graph_partition SYSTEM PRIVATE
        "${ASCEND_TOOLKIT_HOME}/include"
    )

    find_library(HCCL_LIBRARY
        NAMES hccl
        HINTS
            "${ASCEND_TOOLKIT_HOME}/lib64"
            "${ASCEND_TOOLKIT_HOME}/hccl/lib64"
            "${ASCEND_TOOLKIT_HOME}/runtime/lib64/stub"
    )
    find_library(ASCENDCL_LIBRARY
        NAMES ascendcl
        HINTS
            "${ASCEND_TOOLKIT_HOME}/lib64"
            "${ASCEND_TOOLKIT_HOME}/runtime/lib64"
            "${ASCEND_TOOLKIT_HOME}/runtime/lib64/stub"
    )

    if (NOT HCCL_LIBRARY)
        message(FATAL_ERROR "Could not find libhccl. Set ASCEND_TOOLKIT_HOME or HCCL_LIBRARY.")
    endif()
    if (NOT ASCENDCL_LIBRARY)
        message(FATAL_ERROR "Could not find libascendcl. Set ASCEND_TOOLKIT_HOME or ASCENDCL_LIBRARY.")
    endif()

    target_link_libraries(hccl_csr_graph_partition PRIVATE
        "${HCCL_LIBRARY}"
        "${ASCENDCL_LIBRARY}"
    )
endif()


### 5.2 运行脚本

`run_simulate.sh`负责关闭HCCL选项、构建并运行模拟后端。`run_hccl.sh`接收Rank数量，构建HCCL版本，清理RootInfo文件，启动`0..WORLD_SIZE-1`个进程，并把每个Rank的输出保存到`results/hccl_rank_*.log`。


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/scripts/run_simulate.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
PROJECT_DIR="$(cd "${SCRIPT_DIR}/.." && pwd)"

cd "${PROJECT_DIR}"
mkdir -p results

cmake -S . -B build -DENABLE_HCCL=OFF
cmake --build build

./build/hccl_csr_graph_partition \
  --backend simulate \
  --output results/simulate_latest.txt \
  "$@"


In [ ]:
%%writefile src/03.04_extra_hccl_csr_graph_partition/scripts/run_hccl.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
PROJECT_DIR="$(cd "${SCRIPT_DIR}/.." && pwd)"

WORLD_SIZE="${1:-4}"
if [[ $# -gt 0 ]]; then
  shift
fi

ROOT_INFO_FILE="${ROOT_INFO_FILE:-/tmp/hccl_csr_graph_partition_root.info}"
BUILD_DIR="${BUILD_DIR:-build-hccl}"

cd "${PROJECT_DIR}"
mkdir -p results
rm -f "${ROOT_INFO_FILE}"
rm -f results/hccl_rank_*.log results/hccl_rank0_latest.txt

cmake -S . -B "${BUILD_DIR}" -DENABLE_HCCL=ON
cmake --build "${BUILD_DIR}"

pids=()
for rank in $(seq 0 $((WORLD_SIZE - 1))); do
  (
    export RANK="${rank}"
    export WORLD_SIZE="${WORLD_SIZE}"
    export LOCAL_RANK="${rank}"
    "./${BUILD_DIR}/hccl_csr_graph_partition" \
      --backend hccl \
      --world-size "${WORLD_SIZE}" \
      --rank "${rank}" \
      --local-device "${rank}" \
      --root-info-file "${ROOT_INFO_FILE}" \
      --output results/hccl_rank0_latest.txt \
      "$@" \
      > "results/hccl_rank_${rank}.log" 2>&1
  ) &
  pids+=("$!")
done

status=0
for pid in "${pids[@]}"; do
  if ! wait "${pid}"; then
    status=1
  fi
done

if [[ -f results/hccl_rank_0.log ]]; then
  cat results/hccl_rank_0.log
fi

exit "${status}"


In [ ]:
!chmod +x src/03.04_extra_hccl_csr_graph_partition/scripts/run_simulate.sh
!chmod +x src/03.04_extra_hccl_csr_graph_partition/scripts/run_hccl.sh
!find src/03.04_extra_hccl_csr_graph_partition -maxdepth 3 -type f | sort


### 5.3 运行simulate后端

下面使用`simulate`后端运行双Rank逻辑。该命令不依赖CANN和NPU，会验证CSR图生成、三种分片策略、局部度数、补齐、ReduceScatter语义、AllGather语义和串行参考结果。


In [ ]:
!cd src/03.04_extra_hccl_csr_graph_partition && bash scripts/run_simulate.sh \
  --world-size 2 \
  --vertex-count 1024 \
  --edge-count 4000 \
  --seed 2026 \
  --graph-mode skewed \
  --partition-strategy degree_balance \
  --output results/simulate_notebook.txt \
  | tee results/simulate_notebook_console.txt


### 5.4 自动选择HCCL或simulate后端

本单元根据当前环境自动选择运行方式：两张及以上可见NPU使用真实HCCL双Rank，一张可见NPU使用真实HCCL单Rank，没有可用NPU或CANN环境时使用simulate双Rank。若HCCL环境或运行过程失败，会打印失败原因并回退到simulate双Rank，保证Notebook能够继续完成正确性验证。

单Rank HCCL只能验证真实HCCL初始化、设备内存和集合通信API调用，不代表跨设备双Rank通信；只有检测到至少两张NPU并且真实HCCL运行成功时，才记录为双Rank HCCL结果。


In [ ]:
import os
import re
import shlex
import shutil
import subprocess
from pathlib import Path


def find_cann_env_script():
    """Find a CANN set_env.sh without assuming one fixed installation path."""
    candidates = []
    for name in ["CANN_ENV_SCRIPT", "ASCEND_INSTALL_PATH", "ASCEND_TOOLKIT_HOME", "ASCEND_HOME_PATH"]:
        value = os.environ.get(name)
        if not value:
            continue
        path = Path(value)
        if path.name == "set_env.sh":
            candidates.append(path)
        candidates.extend([
            path / "set_env.sh",
            path / "latest" / "set_env.sh",
            path / f"{os.uname().machine}-linux" / "set_env.sh",
        ])

    roots = [Path("/opt/conda/Ascend"), Path("/usr/local/Ascend"), Path("/opt/Ascend"), Path.home() / "Ascend"]
    for root in roots:
        if not root.exists():
            continue
        find_result = subprocess.run(
            ["find", str(root), "-type", "f", "-name", "set_env.sh", "-print", "-quit"],
            capture_output=True,
            text=True,
            check=False,
        )
        if find_result.stdout.strip():
            candidates.append(Path(find_result.stdout.strip()))

    for candidate in candidates:
        if candidate.is_file():
            return candidate
    return None


def detect_visible_npu_count():
    """Detect devices visible to this process, respecting container visibility variables."""
    for name in ["ASCEND_RT_VISIBLE_DEVICES", "ASCEND_VISIBLE_DEVICES"]:
        value = os.environ.get(name)
        if value is None or not value.strip():
            continue
        if value.strip() in {"-1", "none", "None"}:
            return 0
        visible_count = 0
        for item in re.split(r"[,;]", value):
            item = item.strip()
            if not item or item == "-1":
                continue
            range_match = re.fullmatch(r"(\d+)\s*-\s*(\d+)", item)
            if range_match:
                start, end = map(int, range_match.groups())
                visible_count += abs(end - start) + 1
            else:
                visible_count += 1
        return visible_count

    npu_smi = shutil.which("npu-smi")
    if npu_smi is None:
        common_npu_smi = Path("/usr/local/Ascend/driver/tools/npu-smi")
        if common_npu_smi.is_file():
            npu_smi = str(common_npu_smi)
    if npu_smi:
        result = subprocess.run([npu_smi, "info", "-l"], capture_output=True, text=True, check=False)
        text = result.stdout + result.stderr
        for pattern in [
            r"Total\s+number\s+of\s+NPU\s*:\s*(\d+)",
            r"Total\s+NPU\s*(?:number|count)?\s*:\s*(\d+)",
            r"Total\s+Count\s*:\s*(\d+)",
        ]:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                return int(match.group(1))
        ids = set(re.findall(r"NPU\s*ID\s*:\s*(\d+)", text, re.IGNORECASE))
        if ids:
            return len(ids)

    device_path = Path("/dev")
    if device_path.exists():
        return sum(1 for item in device_path.glob("davinci*") if re.fullmatch(r"davinci\d+", item.name))
    return 0


CANN_ENV_SCRIPT = find_cann_env_script()
NPU_COUNT = detect_visible_npu_count()


def run_backend(backend, world_size, output_name, console_name, extra_args):
    """Run one backend and save the complete console output for later inspection."""
    args = list(extra_args) + ["--output", output_name]
    if backend == "hccl":
        if CANN_ENV_SCRIPT is None:
            print("HCCL skipped: CANN set_env.sh was not found.")
            return 2
        shell_command = (
            f"source {shlex.quote(str(CANN_ENV_SCRIPT))} && "
            f"bash scripts/run_hccl.sh {world_size} "
            f"{' '.join(shlex.quote(item) for item in args)}"
        )
        command = ["bash", "-lc", shell_command]
    else:
        command = ["bash", "scripts/run_simulate.sh"] + args

    completed = subprocess.run(
        command,
        cwd=WORK_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    print(completed.stdout)
    (WORK_DIR / "results" / console_name).write_text(completed.stdout, encoding="utf-8")
    return completed.returncode


print("detected_npu_count:", NPU_COUNT)
print("cann_env_script:", CANN_ENV_SCRIPT if CANN_ENV_SCRIPT else "not found")

if NPU_COUNT >= 2 and CANN_ENV_SCRIPT is not None:
    selected_backend = "hccl"
    selected_world_size = 2
    fallback_reason = ""
elif NPU_COUNT == 1 and CANN_ENV_SCRIPT is not None:
    selected_backend = "hccl"
    selected_world_size = 1
    fallback_reason = "only one visible NPU; this is a single-rank HCCL validation"
else:
    selected_backend = "simulate"
    selected_world_size = 2
    fallback_reason = "fewer than one usable NPU or CANN environment unavailable"

print("selected_backend:", selected_backend)
print("selected_world_size:", selected_world_size)
print("fallback_reason:", fallback_reason or "none")

common_args = [
    "--vertex-count", "1024",
    "--edge-count", "4000",
    "--seed", "2026",
    "--graph-mode", "skewed",
    "--partition-strategy", "degree_balance",
]
actual_backend = selected_backend
actual_world_size = selected_world_size
return_code = run_backend(
    selected_backend,
    selected_world_size,
    "results/hccl_auto_latest.txt" if selected_backend == "hccl" else "results/simulate_auto_latest.txt",
    "hccl_auto_notebook.txt" if selected_backend == "hccl" else "simulate_auto_notebook.txt",
    common_args,
)

if selected_backend == "hccl" and return_code != 0:
    print("HCCL execution failed; falling back to simulate with world_size=2.")
    print("fallback_reason: HCCL command returned", return_code)
    actual_backend = "simulate"
    actual_world_size = 2
    return_code = run_backend(
        "simulate",
        2,
        "results/simulate_auto_latest.txt",
        "simulate_auto_notebook.txt",
        common_args,
    )

print("actual_backend:", actual_backend)
print("actual_world_size:", actual_world_size)
print("run_return_code:", return_code)
if return_code != 0:
    raise RuntimeError(f"Both the selected backend and fallback failed, return_code={return_code}")


### 5.5 自动选择双Rank多策略对照实验

两张及以上可见NPU且CANN环境可用时，本单元运行四组真实HCCL双Rank对照；少于两张NPU、CANN不可用或某一组HCCL运行失败时，自动使用simulate双Rank完成相同四组策略对照。模拟结果仍可用于验证分片负载、度数汇总和正确性，但不代表真实跨设备通信耗时。


In [ ]:
cases = [
    ("uniform", "vertex_range"),
    ("skewed", "vertex_range"),
    ("skewed", "degree_balance"),
    ("skewed", "edge_hash"),
]

strategy_backend = "hccl" if NPU_COUNT >= 2 and CANN_ENV_SCRIPT is not None else "simulate"
print("strategy_backend:", strategy_backend)
print("strategy_world_size: 2")

for graph_mode, strategy in cases:
    label = f"{graph_mode}_{strategy}"
    print("=====", label, "=====")
    common_args = [
        "--vertex-count", "1024",
        "--edge-count", "4000",
        "--seed", "2026",
        "--graph-mode", graph_mode,
        "--partition-strategy", strategy,
    ]
    output_name = f"results/{'hccl' if strategy_backend == 'hccl' else 'simulate'}_w2_{label}.txt"
    console_name = f"{'hccl' if strategy_backend == 'hccl' else 'simulate'}_w2_{label}_console.txt"
    return_code = run_backend(
        strategy_backend,
        2,
        output_name,
        console_name,
        common_args,
    )
    if strategy_backend == "hccl" and return_code != 0:
        print("HCCL case failed; rerunning this case with simulate backend.")
        output_name = f"results/simulate_w2_{label}.txt"
        console_name = f"simulate_w2_{label}_console.txt"
        return_code = run_backend("simulate", 2, output_name, console_name, common_args)
    print("case_return_code:", return_code)
    if return_code != 0:
        raise RuntimeError(f"Strategy case {label} failed in both HCCL and simulate backends")


### 5.6 读取实验结果

模拟后端将精简摘要写入`simulate_notebook.txt`、自动回退的`simulate_auto_latest.txt`或策略对照结果文件；真实HCCL后端由Rank 0写入相应的`hccl_*.txt`文件，并为每个Rank保存日志。下面读取当前目录中可用的摘要文件，重点观察后端、分片策略、ReduceScatter分片长度、通信时间和正确性字段。


In [ ]:
from pathlib import Path

print("detected_npu_count:", globals().get("NPU_COUNT", "not detected"))
print("actual_backend:", globals().get("actual_backend", "run 5.4 first"))
print("actual_world_size:", globals().get("actual_world_size", "run 5.4 first"))

summary_paths = [
    WORK_DIR / "results" / "simulate_notebook.txt",
    WORK_DIR / "results" / "simulate_auto_latest.txt",
    WORK_DIR / "results" / "hccl_auto_latest.txt",
    WORK_DIR / "results" / "hccl_rank0_latest.txt",
]
summary_paths.extend(sorted((WORK_DIR / "results").glob("*_w2_*.txt")))

seen = set()
for path in summary_paths:
    if path in seen:
        continue
    seen.add(path)
    print("=", path)
    if path.exists():
        print(path.read_text(encoding="utf-8", errors="ignore")[:5000])
    else:
        print("not found")


### 5.7 正确性验证

正确性验证重点观察：

1. `result=PASS`且`mismatch_count=0`，表示AllGather恢复出的全局度数向量与串行CSR参考结果逐元素一致；
2. `padded_vertex_count`能够被`world_size`整除，且`shard_size=padded_vertex_count/world_size`；
3. 所有ReduceScatter分片的`shard_sum`之和等于`adjacency_count=2*edge_count`；
4. 全局Top-K顶点及度数在模拟后端和HCCL后端之间一致；
5. `item_imbalance`与`adjacency_imbalance`应符合当前分片策略的预期；
6. 只有`actual_backend=hccl`且`actual_world_size=2`时，才可以将结果解释为真实双Rank HCCL结果。


### 5.8 双Rank策略对比分析

参考双Rank实验使用1024个顶点、4000条边和2026随机种子。均匀图下，`vertex_range`的邻接项负载为3926和4074，`adjacency_imbalance=1.018`，已经比较均衡；倾斜图下，`vertex_range`的邻接项负载为5666和2334，`adjacency_imbalance=1.417`，说明低编号热点顶点集中在Rank 0。

`degree_balance`在倾斜图下将邻接项控制为4000和4000，虽然顶点数为518和506、`item_imbalance=1.012`，但更接近实际CSR扫描工作量均衡。`edge_hash`将两边分配为2000和2000、邻接项为4000和4000，但同一顶点的度数贡献可能分散到两个Rank，因此更依赖ReduceScatter的全局求和。


### 5.9 ReduceScatter分片与通信缓冲区分析

ReduceScatter分片按照全局顶点编号区间切分，并不等同于前面选择的计算分片。对三组`skewed`实验，无论使用`vertex_range`、`degree_balance`还是`edge_hash`，全局度数分布相同，因此两个分片的度数总和仍分别为5666和2334；变化的是各Rank如何产生局部贡献。

双Rank默认配置下，每个Rank输入1024个`int64`，即8192 bytes；ReduceScatter后保留512个`int64`，即4096 bytes；AllGather后恢复8192 bytes。四组对照的通信缓冲区规模完全相同，因此策略差异主要体现在局部计算负载，而不是通信消息长度。


### 5.10 通信耗时分析





In [ ]:
import re

def read_summary(path):
    values = {}
    if not path.exists():
        return values
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        if ":" in line and not line.startswith("  "):
            key, value = line.split(":", 1)
            values[key.strip()] = value.strip()
    return values

for path in sorted((WORK_DIR / "results").glob("*.txt")):
    values = read_summary(path)
    if not values:
        continue
    print("=", path.name)
    for key in [
        "backend",
        "world_size",
        "graph_mode",
        "partition_strategy",
        "padded_vertex_count",
        "shard_size",
        "reduce_scatter_ms",
        "all_gather_ms",
        "result",
        "mismatch_count",
        "item_imbalance",
        "adjacency_imbalance",
    ]:
        if key in values:
            print(f"{key}: {values[key]}")


---
## 6. 实验总结

本实验按照实验概述、环境准备、问题分析、核心程序开发和结果验证与性能分析五个阶段，实现了基于HCCL的CSR图分片与全局度数统计。

* CSR使用`row_ptr`和`col_indices`紧凑保存不规则邻接表，一条无向边展开为两个邻接项。
* `vertex_range`实现简单的连续顶点分片，`edge_hash`实现边级分散，`degree_balance`利用度数信息均衡邻接项工作量。
* 每个Rank生成全局长度的局部度数贡献向量，通过ReduceScatter完成全局求和和连续分片，再由AllGather恢复完整向量。
* `simulate`后端用于验证应用逻辑，真实`hccl`后端通过多进程、设备绑定和RootInfo文件建立HCCL通信域。
* 倾斜图下，按顶点编号切分可能造成热点集中；按度数均衡能够改善CSR扫描负载，按边哈希能够均衡边处理但会分散顶点贡献。
* 正确性分析应关注`PASS`、`mismatch_count=0`和Top-K一致性；性能分析还应结合通信缓冲区规模和多次重复运行结果。

通过本实验，可以理解不规则图从CSR压缩存储、分片计算到HCCL全局汇总的完整流程，掌握分片策略与负载均衡之间的关系，并具备分析真实多Rank集合通信结果的能力。
